# CLTK Evaluation on Glosses
This notebook tests LatinCy for lemmatization and POS tagging on ~600 Latin glosses. Results are compared to the dataset's original tags.

Created by Thea Schaaf, March 2025

In [5]:
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

Set os path environment for opening files
Configuration of sample names and tasks, results storage

In [6]:
# Configuration
MODEL_NAME = "CLTK"
SAMPLE_TYPES = ["medieval_charters", "glosses"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("03-cltk.ipynb")


In [7]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

Import code

In [8]:
from cltk.data.fetch import FetchCorpus
fetcher = FetchCorpus(language="lat")
fetcher.import_corpus("lat_models_cltk")

from cltk.nlp import NLP


In [9]:
cltk_nlp = NLP(language="lat")


‎𐤀 CLTK version '1.5.0'. When using the CLTK in research, please cite: https://aclanthology.org/2021.acl-demo.3/

Pipeline for language 'Latin' (ISO: 'lat'): `LatinNormalizeProcess`, `LatinStanzaProcess`, `LatinEmbeddingsProcess`, `StopsProcess`, `LatinLexiconProcess`.

⸖ ``LatinStanzaProcess`` using Stanza model from the Stanford NLP Group: https://stanfordnlp.github.io/stanza/ . Please cite: https://arxiv.org/abs/2003.07082
⸖ ``LatinEmbeddingsProcess`` using word2vec model by University of Oslo from http://vectors.nlpl.eu/ . Please cite: https://aclanthology.org/W17-0237/
⸖ ``LatinLexiconProcess`` using Lewis's *An Elementary Latin Dictionary* (1890).

⸎ To suppress these messages, instantiate ``NLP()`` with ``suppress_banner=True``.


In [10]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [ ]:
def analyse(df, sample_type, df_version):
    # Convert the lemma columns to string type to ensure consistent comparison
    df['lemma_gold'] = df['lemma_gold'].astype(str)
    df['lemma_pred'] = df['lemma_pred'].astype(str)

    # Evaluate lemmatization
    lemma_accuracy = accuracy_score(df['lemma_gold'], df['lemma_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['lemma_gold'] == df['lemma_pred']

    # Fix the precision_recall_fscore_support call
    lemma_precision, lemma_recall, lemma_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    # Convert the lemma columns to string type to ensure consistent comparison
    df['pos_gold'] = df['pos_gold'].astype(str)
    df['pos_pred'] = df['pos_pred'].astype(str)

    # Evaluate lemmatization
    pos_accuracy = accuracy_score(df['pos_gold'], df['pos_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = df['pos_gold'] == df['pos_pred']

    # Fix the precision_recall_fscore_support call
    pos_precision, pos_recall, pos_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(df),
        average='binary'
    )

    results["accuracy"][f"{sample_type}_{df_version}_lemma"] = lemma_accuracy
    results["precision"][f"{sample_type}_{df_version}_lemma"] = lemma_precision
    results["recall"][f"{sample_type}_{df_version}_lemma"] = lemma_recall
    results["f1_score"][f"{sample_type}_{df_version}_lemma"] = lemma_f1

    results["accuracy"][f"{sample_type}_{df_version}_pos"] = pos_accuracy
    results["precision"][f"{sample_type}_{df_version}_pos"] = pos_precision
    results["recall"][f"{sample_type}_{df_version}_pos"] = pos_recall
    results["f1_score"][f"{sample_type}_{df_version}_pos"] = pos_f1

    df.to_csv(f"../results/{MODEL_NAME}_{sample_type}_{df_version}_detailed.csv", index=False)

    print(f"Completed {sample_type} {df_version}. Processing time: {processing_time:.2f}s")
    print(f"Lemmatization {df_version} accuracy: {lemma_accuracy:.4f}")
    print(f"POS tagging {df_version} accuracy: {pos_accuracy:.4f}")
    print("-" * 50)



In [12]:
def cltk_processor(sample_texts, sample_type):
    processed_results = []
    for sample_id, text in sample_texts:
        doc = cltk_nlp.analyze(text=text)
        for sent in doc.sentences:
            for idx, word in enumerate(sent.words):
                if sample_type == "glosses":
                    word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{idx}"
                else:
                    word_id = str(idx)
                processed_results.append({
                    "sample_id": sample_id,
                    "word_id": word_id,
                    "word": word.string,
                    "lemma": word.lemma,
                    "pos": word.upos,
                })

    return processed_results


In [ ]:
for sample_type in SAMPLE_TYPES:
    print(f"Processing {sample_type}...")

    # Load gold standard data
    gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

    gold_df = pd.read_csv(gold_file)

    # Extract text to reconstruct from words
    sample_texts = word_joiner(gold_df)

    # Process samples and measure time
    start_time = time.time()

    processed_results = cltk_processor(sample_texts, sample_type)

    processing_time = time.time() - start_time
    results["processing_times"][sample_type] = processing_time

    print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

    pred_df = pd.DataFrame(processed_results)

    # Add a token index per sample in both gold and pred dataframes
    gold_df['token_idx'] = gold_df.groupby('sample_id').cumcount()
    pred_df['token_idx'] = pred_df.groupby('sample_id').cumcount()

    # Merge on sample_id and token index
    merged_df = pd.merge(gold_df, pred_df, on=['sample_id', 'token_idx'], suffixes=('_gold', '_pred'))

    # Check mismatches
    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']

    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']
    mismatched_df = merged_df[~merged_df['word_match']].head()
    print(len(mismatched_df))

   # Create df version without mismatches
    aligned_df = merged_df[merged_df['word_match']].copy()

    print(f"Running analysis on {sample_type}...")
    analyse(merged_df, sample_type, "all")
    analyse(aligned_df, sample_type, "aligned_only")




Processing medieval_charters...
Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Variant' with value 'Greek'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If

KeyboardInterrupt: 

In [42]:
# Save summary results
with open(f"../results/{MODEL_NAME}_summary.json", "w") as f:
    json.dump(results, f, indent=2)
